In [1]:
from pathlib import Path
import xmltodict
import pandas as pd
import numpy as np
import tifffile as tiff
import xmltodict
from dateutil import parser
import re

import matplotlib.pyplot as plt
from collections import namedtuple
from shapely.geometry import box

from skimage.registration import phase_cross_correlation
from skimage.transform import rescale, resize, downscale_local_mean
from skimage import exposure

import zarr
from pylibCZIrw import czi as pyczi

In [ ]:
"""Skip during test
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Intermediate")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Proximal")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\TA23\Distal")
section_folders = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_dir() and folder.name.startswith("S_"):
        print(f"fould series folder: {folder.name}")
        section_folders.append(folder)
"""

In [2]:
from atlas.io import is_there_a_single_tif, extract_s_number

In [16]:
series_folder = Path(r"E:\PROJECTS\EM\zeinab\ATLAS-projects\L32-10-2-w2-ROI-20nm-BSD")
series_folder = Path(r"E:\PROJECTS\EM\zeinab\ATLAS-projects\L32-10-1-ROI-w1-20nm-bsd")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\ATLAS-projects\TA31-atlas\Stitched-Sets\TA31-20nm-roi1-2")
series_folder = Path(r"E:\PROJECTS\EM\LUKE\ATLAS-projects\TA31-atlas\Stitched-Sets\TA31-20nm-roi3")
series_folder = Path(r"E:\PROJECTS\EM\zeinab\ATLAS-projects\atlas_olive_2\roi-01-stitched")
series_folder = Path(r"E:\PROJECTS\EM\Filipa\M2-2\Filipa-M2-2-20251127\exports\Tissue-overview-2")
series_folder = Path(r"E:\PROJECTS\EM\Alma\plant-3-wafer-1\alma-plant-3_data\session_422002753\Site 2")
series_folder = Path(r"E:\PROJECTS\EM\Alma\plant4-wafer-1\alma-plant-4_data\session_1084385689\Site 1")
series_folder = Path(r"E:\PROJECTS\EM\Algae Project\Meso\Meso_data\session_1152502930\Site 2")
series_folder = Path(r"E:\PROJECTS\EM\Filipa\M2-2\ATLAS-projects\proj-4-20260302\NEW LOCATION20260303_data\session_1149164643\Site 1")
series_folder = Path(r"E:\PROJECTS\EM\Thomas Gerner\Mice tisssue project_data\session_239138424\Section Set 2")

# Find the first CSV file that starts with "PhaseCC_stitching_S_"
csv_file = next(series_folder.glob("PhaseCC_stitching_S_*.csv"), None)

if csv_file is None:
    raise FileNotFoundError("No matching CSV file found in the folder.")

print(f"Found CSV file: {csv_file.name}")

# Load the CSV into a DataFrame
df = pd.read_csv(csv_file)

# Try to find a 'pixelSize' column or entry
if 'PixelSizeMicron' in df.columns:
    pix_size_micron =  df['PixelSizeMicron'].max()
else:
    raise ValueError("No 'pixelSize' parameter found in the CSV.")

print(f"pixel_size in microns {pix_size_micron}")


#TODO: important change this to metadata readout
pixel_size = {
    'Value': pix_size_micron,
    'Axial': 0.300,
    'Unit': 'µm'
}
print("work on pixel size")
""" EXAMPLE
with open(tif_folder.joinpath('tif-stack-metadata.xml'), "rb") as f:
    metadata_dict = xmltodict.parse(f, xml_attribs=True)

pixel_size = metadata_dict['human_meta_data']['pixel_size']
"""

tif_list = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_file() and folder.name.endswith(".tiff"):
        print(f"fould series image: {folder.name}")
        tif_list.append(folder)

Found CSV file: phaseCC_stitching_S_1.csv
pixel_size in microns 0.0149977765977383
work on pixel size
fould series image: stitched_image_S_1.tiff
fould series image: stitched_image_S_10.tiff
fould series image: stitched_image_S_11.tiff
fould series image: stitched_image_S_12.tiff
fould series image: stitched_image_S_13.tiff
fould series image: stitched_image_S_14.tiff
fould series image: stitched_image_S_15.tiff
fould series image: stitched_image_S_16.tiff
fould series image: stitched_image_S_17.tiff
fould series image: stitched_image_S_18.tiff
fould series image: stitched_image_S_19.tiff
fould series image: stitched_image_S_2.tiff
fould series image: stitched_image_S_20.tiff
fould series image: stitched_image_S_21.tiff
fould series image: stitched_image_S_22.tiff
fould series image: stitched_image_S_23.tiff
fould series image: stitched_image_S_24.tiff
fould series image: stitched_image_S_25.tiff
fould series image: stitched_image_S_26.tiff
fould series image: stitched_image_S_27.tiff


In [17]:
"""SKIP during test
tif_list = []
for section_folder in section_folders:
    single_tif, tif_path = is_there_a_single_tif(section_folder)
    if single_tif:
        #print(tif_path)
        tif_list.append(tif_path)
"""

tif_list_sorted = sorted(tif_list, key=extract_s_number)
from atlas.io import reorder_files_by_s_number, parse_shorthand_order

In [18]:
correct_order_path = Path(series_folder.joinpath("correct_order.txt"))

try:
    new_order = parse_shorthand_order(correct_order_path)
    print(f"Loaded correct_order.txt: {new_order}")

    tif_list_sorted = reorder_files_by_s_number(tif_list_sorted, new_order)
    print("Files reordered based on correct_order.txt:")
    for f in tif_list_sorted:
        print(f)

except FileNotFoundError:
    print("No correct_order.txt found — using original order.")
except Exception as e:
    print(f"Error reading or applying correct_order.txt: {e}")

No correct_order.txt found — using original order.


In [19]:
from atlas.io.fibics_metadata import extract_tif_metadata, get_pixel_size_from_tif
from atlas.io import create_empty_folder, rm_tree, apply_alignment, zarr_array_to_czi
from atlas.image_analysis import image_dtype_min_max, rescale_image_intensity
from atlas.alignment import calculate_cumulative_shifts, initialize_alignment_df, pairwise_alignment, first_last_true, ROI
from atlas.image_analysis import image_dtype_min_max, mask_low_and_saturation, rescale_image_intensity


In [20]:
output_path = series_folder.joinpath("alignment_results")
create_empty_folder(output_path)

down_scale = 10

# initialize the dataframe, in particular we asign the pair-wise patching of the images
# based on the sorted list of tiff.
z_align_df = initialize_alignment_df(tif_list_sorted, down_scale)
# run pairwise alignment based on the provided dataframe
z_align_df = pairwise_alignment(z_align_df)
# now we can calculate the cumulative shifts
z_align_df = calculate_cumulative_shifts(z_align_df)

# ✅ At the end, save the DataFrame as a CSV for later analysis
z_align_df_path = output_path.joinpath("z_alignment_results.pkl")
z_align_df.to_pickle(z_align_df_path)

z_align_df 

#TODO: implement handling of outliers in the alignment results, this can be done by looking at the current shift and then decide whichones looks weird.

Created new (or emptied) folder: E:\PROJECTS\EM\Thomas Gerner\Mice tisssue project_data\session_239138424\Section Set 2\alignment_results
Processing alignment: ref -> stitched_image_S_1.tiff, moving -> stitched_image_S_1.tiff
crop pixel shift: [0 0]
Detected pixel offset based on crops (row, col): [0. 0.]
current shift: [0. 0.]
Processing alignment: ref -> stitched_image_S_1.tiff, moving -> stitched_image_S_2.tiff
crop pixel shift: [0 0]
Detected pixel offset based on crops (row, col): [90. 83.]
current shift: [90. 83.]
Processing alignment: ref -> stitched_image_S_2.tiff, moving -> stitched_image_S_3.tiff
crop pixel shift: [0 0]
Detected pixel offset based on crops (row, col): [120.  28.]
current shift: [120.  28.]
Processing alignment: ref -> stitched_image_S_3.tiff, moving -> stitched_image_S_4.tiff
crop pixel shift: [0 0]
Detected pixel offset based on crops (row, col): [-95.  -5.]
current shift: [-95.  -5.]
Processing alignment: ref -> stitched_image_S_4.tiff, moving -> stitched_i

,moving_path,reference_path,moving_ROI,reference_ROI,current_shift,cumulative_ROI,cumulative_shift,down_scale
0,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1685, 0, 3759)","(0, 1685, 0, 3759)","[0.0, 0.0]","(0, 1685, 0, 3759)","[0.0, 0.0]",10
1,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1567, 0, 3676)","(0, 1685, 0, 3759)","[90.0, 83.0]","(83, 1650, 90, 3766)","[90.0, 83.0]",10
2,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1507, 0, 3514)","(0, 1567, 0, 3676)","[120.0, 28.0]","(111, 1618, 210, 3724)","[210.0, 111.0]",10
3,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1550, 0, 3816)","(0, 1507, 0, 3514)","[-95.0, -5.0]","(106, 1656, 115, 3931)","[115.0, 106.0]",10
4,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1532, 0, 3737)","(0, 1550, 0, 3816)","[113.0, -86.0]","(20, 1552, 228, 3965)","[228.0, 20.0]",10
5,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1528, 0, 3627)","(0, 1532, 0, 3737)","[-16.0, -2.0]","(18, 1546, 212, 3839)","[212.0, 18.0]",10
6,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1553, 0, 3740)","(0, 1528, 0, 3627)","[-134.0, 49.0]","(67, 1620, 78, 3818)","[78.0, 67.0]",10
7,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1574, 0, 3870)","(0, 1553, 0, 3740)","[-473.0, -92.0]","(-25, 1549, -395, 3475)","[-395.0, -25.0]",10
8,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1544, 0, 3465)","(0, 1574, 0, 3870)","[590.0, 188.0]","(163, 1707, 195, 3660)","[195.0, 163.0]",10
9,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1493, 0, 3738)","(0, 1544, 0, 3465)","[1.0, -69.0]","(94, 1587, 196, 3934)","[196.0, 94.0]",10


In [21]:
z_align_df = pd.read_pickle(z_align_df_path)
z_align_df

,moving_path,reference_path,moving_ROI,reference_ROI,current_shift,cumulative_ROI,cumulative_shift,down_scale
0,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1685, 0, 3759)","(0, 1685, 0, 3759)","[0.0, 0.0]","(0, 1685, 0, 3759)","[0.0, 0.0]",10
1,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1567, 0, 3676)","(0, 1685, 0, 3759)","[90.0, 83.0]","(83, 1650, 90, 3766)","[90.0, 83.0]",10
2,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1507, 0, 3514)","(0, 1567, 0, 3676)","[120.0, 28.0]","(111, 1618, 210, 3724)","[210.0, 111.0]",10
3,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1550, 0, 3816)","(0, 1507, 0, 3514)","[-95.0, -5.0]","(106, 1656, 115, 3931)","[115.0, 106.0]",10
4,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1532, 0, 3737)","(0, 1550, 0, 3816)","[113.0, -86.0]","(20, 1552, 228, 3965)","[228.0, 20.0]",10
5,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1528, 0, 3627)","(0, 1532, 0, 3737)","[-16.0, -2.0]","(18, 1546, 212, 3839)","[212.0, 18.0]",10
6,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1553, 0, 3740)","(0, 1528, 0, 3627)","[-134.0, 49.0]","(67, 1620, 78, 3818)","[78.0, 67.0]",10
7,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1574, 0, 3870)","(0, 1553, 0, 3740)","[-473.0, -92.0]","(-25, 1549, -395, 3475)","[-395.0, -25.0]",10
8,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1544, 0, 3465)","(0, 1574, 0, 3870)","[590.0, 188.0]","(163, 1707, 195, 3660)","[195.0, 163.0]",10
9,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,E:\PROJECTS\EM\Thomas Gerner\Mice tisssue proj...,"(0, 1493, 0, 3738)","(0, 1544, 0, 3465)","[1.0, -69.0]","(94, 1587, 196, 3934)","[196.0, 94.0]",10


In [22]:
use_down_sample = True
zarr_array, zarr_path = apply_alignment(z_align_df, buffer_pixels=20, percentile_low=2.0, percentile_high=99.5, use_down_sample=use_down_sample)

out_pixel_size = pixel_size.copy()

if use_down_sample:
    out_pixel_size['Value'] = pixel_size['Value'] * down_scale
    end_str = "_ds_aligned"
else:
    end_str = "_aligned"

czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

we will downsample during saving of the zarr, this is good during testing for visual inspection
shape of the zarr array to create: (4687, 2070, 30)
creating zarr at: E:\PROJECTS\EM\Thomas Gerner\Mice tisssue project_data\session_239138424\Section Set 2\Section Set 2.zarr
Done for idx 0
Done for idx 1
Done for idx 2
Done for idx 3
Done for idx 4
Done for idx 5
Done for idx 6
Done for idx 7
Done for idx 8
Done for idx 9
Done for idx 10
Done for idx 11
Done for idx 12
Done for idx 13
Done for idx 14
Done for idx 15
Done for idx 16
Done for idx 17
Done for idx 18
Done for idx 19
Done for idx 20
Done for idx 21
Done for idx 22
Done for idx 23
Done for idx 24
Done for idx 25
Done for idx 26
Done for idx 27
Done for idx 28
Done for idx 29
Saving CZI file to: E:\PROJECTS\EM\Thomas Gerner\Mice tisssue project_data\session_239138424\Section Set 2\Section Set 2_ds_aligned.czi
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processi

In [23]:
if zarr_path.exists():
  rm_tree(zarr_path)

In [24]:
use_down_sample = False
zarr_array, zarr_path = apply_alignment(z_align_df, buffer_pixels=1, percentile_low=2.0, percentile_high=99.8, use_down_sample=use_down_sample)

doing full scale alignment, it is recomended to check results before with downscale
shape of the zarr array to create: (46472, 20302, 30)
creating zarr at: E:\PROJECTS\EM\Thomas Gerner\Mice tisssue project_data\session_239138424\Section Set 2\Section Set 2.zarr
Done for idx 0
Done for idx 1
Done for idx 2
Done for idx 3
Done for idx 4
Done for idx 5
Done for idx 6
Done for idx 7
Done for idx 8
Done for idx 9
Done for idx 10
Done for idx 11
Done for idx 12
Done for idx 13
Done for idx 14
Done for idx 15
Done for idx 16
Done for idx 17
Done for idx 18
Done for idx 19
Done for idx 20
Done for idx 21
Done for idx 22
Done for idx 23
Done for idx 24
Done for idx 25
Done for idx 26
Done for idx 27
Done for idx 28
Done for idx 29


In [25]:
out_pixel_size = pixel_size.copy()

if use_down_sample:
    out_pixel_size['Value'] = pixel_size['Value'] * down_scale
    end_str = "_ds_aligned"
else:
    end_str = "_aligned"

czi_path = zarr_array_to_czi(zarr_path, out_pixel_size, end_str=end_str)

Saving CZI file to: E:\PROJECTS\EM\Thomas Gerner\Mice tisssue project_data\session_239138424\Section Set 2\Section Set 2_aligned.czi
Processing frame 0
Processing frame 1
Processing frame 2
Processing frame 3
Processing frame 4
Processing frame 5
Processing frame 6
Processing frame 7
Processing frame 8
Processing frame 9
Processing frame 10
Processing frame 11
Processing frame 12
Processing frame 13
Processing frame 14
Processing frame 15
Processing frame 16
Processing frame 17
Processing frame 18
Processing frame 19
Processing frame 20
Processing frame 21
Processing frame 22
Processing frame 23
Processing frame 24
Processing frame 25
Processing frame 26
Processing frame 27
Processing frame 28
Processing frame 29
